<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
!#pip3 install folium
%pip install wget
%pip install -U ipywidgets
!#pip3 install pandas

"#pip3" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


"#pip3" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [26]:
import folium
import pandas as pd
import base64
from IPython.display import IFrame

# Fondo de OpenStreetMap que sí carga dentro del iframe incrustado
TILES = "OpenStreetMap.DE"

def mostrar_mapa(mapa, height=500):
    """Muestra un mapa de Folium dentro del notebook, evitando el fallo de VS Code
    ("Make this Notebook Trusted to load map")."""
    html = mapa.get_root().render()
    b64 = base64.b64encode(html.encode("utf-8")).decode("utf-8")
    return IFrame(src=f"data:text/html;charset=utf-8;base64,{b64}", width="100%", height=height)


In [ ]:
def mostrar_mapa(mapa, height=500, centro=None, zoom=None):
    """Muestra un mapa de Folium dentro del notebook, evitando el fallo de VS Code.
    centro=[lat, long] y zoom (opcionales) fijan la vista inicial del mapa."""
    if centro is not None or zoom is not None:
        # Quita los fit_bounds anteriores: su zoom animado anularía la vista pedida
        for nombre, hijo in list(mapa._children.items()):
            if isinstance(hijo, folium.map.FitBounds):
                del mapa._children[nombre]
    if centro is not None:
        mapa.location = centro
    if zoom is not None:
        mapa.options["zoom"] = zoom
    html = mapa.get_root().render()
    b64 = base64.b64encode(html.encode("utf-8")).decode("utf-8")
    return IFrame(src=f"data:text/html;charset=utf-8;base64,{b64}", width="100%", height=height)

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [27]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(URL)

Now, you can take a look at what are the coordinates for each site.


In [28]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [17]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [29]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10, tiles=TILES)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [30]:
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(150, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>%s</b></div>' % 'NASA JSC',
    )
)
site_map.add_child(circle)
site_map.add_child(marker)
mostrar_mapa(site_map)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [19]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [ ]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5, tiles=TILES)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True) \
        .add_child(folium.Popup(site['Launch Site'])) \
        .add_to(site_map)
    folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(150, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>%s</b></div>' % site['Launch Site'],
        )
    ).add_to(site_map)

# Encuadra el mapa para que se vean California y Florida a la vez
site_map.fit_bounds([[launch_sites_df['Lat'].min(), launch_sites_df['Long'].min()],
                     [launch_sites_df['Lat'].max(), launch_sites_df['Long'].max()]], padding=(30, 30))

mostrar_mapa(site_map)

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [32]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [33]:
marker_cluster = MarkerCluster()

_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [24]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

In [34]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


In [36]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    # Un marcador nuevo por lanzamiento, coloreado según su resultado
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} | class={record['class']}",
    )
    marker_cluster.add_child(marker)

mostrar_mapa(site_map)

From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [38]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    # Un marcador nuevo por lanzamiento, coloreado según su resultado
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} | class={record['class']}",
    )
    marker_cluster.add_child(marker)

mostrar_mapa(site_map)

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [39]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [41]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Coordenadas del sitio CCAFS SLC-40, tomadas del propio DataFrame
launch_site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
launch_site_lat, launch_site_lon = launch_site['Lat'], launch_site['Long']

# Punto de costa más cercano (leído con MousePosition)
coastline_lat, coastline_lon = 28.56422, -80.56813

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distancia a la costa: {distance_coastline:.2f} km")

Distancia a la costa: 0.86 km


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [ ]:
coastline_coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(100, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>%s</b></div>' % "{:.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker);   # el ";" evita que se muestre el mapa por la vía que falla

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [46]:
coordinates = [[launch_site_lat, launch_site_lon], coastline_coordinate]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

mostrar_mapa(site_map, centro=[launch_site_lat, launch_site_lon], zoom=15)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [47]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
def marcar_proximidad(lat, lon, nombre):
    """Añade un marcador con la distancia al sitio y una línea que los une."""
    distance = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    folium.Marker(
        [lat, lon],
        popup=nombre,
        icon=DivIcon(
            icon_size=(100, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>%s</b></div>' % "{:.2f} KM".format(distance),
        ),
    ).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=1).add_to(site_map)
    print(f"{nombre}: {distance:.2f} km")
    return distance

railway_lat, railway_lon = 28.57113, -80.58543   # NASA Railroad
highway_lat, highway_lon = 28.56385, -80.57086   # Samuel C Phillips Parkway
city_lat, city_lon = 28.61262, -80.80795         # Titusville

distance_railway = marcar_proximidad(railway_lat, railway_lon, "Ferrocarril: NASA Railroad")
distance_highway = marcar_proximidad(highway_lat, highway_lon, "Carretera: Samuel C Phillips Pkwy")
distance_city = marcar_proximidad(city_lat, city_lon, "Ciudad: Titusville")

Ferrocarril: NASA Railroad: 1.22 km
Carretera: Samuel C Phillips Pkwy: 0.59 km
Ciudad: Titusville: 23.23 km


In [48]:
# Vista cercana: costa, carretera y ferrocarril
mostrar_mapa(site_map, centro=[launch_site_lat, launch_site_lon], zoom=15)

In [49]:
# Vista amplia: incluye la ciudad más cercana (Titusville)
mostrar_mapa(site_map, centro=[(launch_site_lat + city_lat) / 2, (launch_site_lon + city_lon) / 2], zoom=11)

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways? 1.22 km
- Are launch sites in close proximity to highways? 0.59 km
- Are launch sites in close proximity to coastline? 0.86 km
- Do launch sites keep certain distance away from cities? 23.23 km

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
